# 03 · Índices físicos y planes de ejecución

**Fase 3 — Tuning.** Se mide con una tabla de prueba (`bench_envios`, 500 000 filas, sin índices)
para no alterar los datos del sistema: primero el plan *sin* índices, luego con ellos, y por último el
costo que pagan las escrituras.

In [ ]:
import pandas as pd
from IPython.display import display

from logistica import indices
from logistica.conexiones import conectar_pg

pd.set_option("display.width", 160)
FILAS = 500_000
conexion = conectar_pg("centro", autocommit=True)
indices.crear_tabla_bench(conexion, FILAS)
indices.quitar_indices(conexion, indices.TABLA_BENCH)
conexion.execute("ANALYZE bench_envios")
print("Tabla de prueba lista:", FILAS, "filas")

## Consultas representativas

| | Consulta | Índice pensado |
|---|---|---|
| Q1 | envíos recientes de un cliente (portal) | B-tree compuesto `(cliente_id, fecha_alta DESC)` |
| Q2 | envíos abiertos hacia una sucursal (panel operativo) | índice **parcial** sobre estados abiertos |
| Q3 | reporte por rango de fechas | **BRIN** sobre `fecha_alta` |

In [ ]:
CONSULTAS = {
    "Q1 cliente": "SELECT * FROM bench_envios WHERE cliente_id = 4321 ORDER BY fecha_alta DESC LIMIT 20",
    "Q2 abiertos": "SELECT count(*) FROM bench_envios WHERE estado IN ('pendiente', 'en_transito') AND sucursal_destino_id = 8",
    "Q3 fechas": "SELECT count(*) FROM bench_envios WHERE fecha_alta >= '2025-03-01' AND fecha_alta < '2025-03-08'",
}
sin_indices = {n: indices.explicar(conexion, q) for n, q in CONSULTAS.items()}
for nombre, plan in sin_indices.items():
    print(f"--- {nombre} (sin índices)\n{plan}\n")

In [ ]:
indices.aplicar_indices(conexion, indices.TABLA_BENCH)
conexion.execute("ANALYZE bench_envios")
con_indices = {n: indices.explicar(conexion, q) for n, q in CONSULTAS.items()}
for nombre, plan in con_indices.items():
    print(f"--- {nombre} (con índices)\n{plan}\n")

In [ ]:
comparacion = pd.DataFrame(
    {
        "sin_indices_ms": {n: indices.tiempo_ejecucion_ms(p) for n, p in sin_indices.items()},
        "con_indices_ms": {n: indices.tiempo_ejecucion_ms(p) for n, p in con_indices.items()},
    }
)
comparacion["mejora_x"] = (comparacion.sin_indices_ms / comparacion.con_indices_ms).round(1)
display(comparacion.round(2))
display(pd.DataFrame(indices.tamanios_indices(conexion, indices.TABLA_BENCH)))

**Lectura.** El B-tree compuesto reemplaza un `Seq Scan` por un `Index Scan` que ya devuelve las filas
ordenadas (sin `Sort`). El índice parcial sólo cubre los envíos abiertos (~2 %), por eso es mucho más
chico que un B-tree completo. El BRIN resuelve el rango de fechas con un tamaño de KB porque la tabla es
*append-only* y `fecha_alta` está físicamente correlacionada.

## Costo sobre las escrituras

In [ ]:
costo = indices.costo_de_escritura(filas_existentes=FILAS, filas_lote=100_000)
display(pd.Series(costo).round(2).to_frame("valor"))

Cada índice B-tree suma trabajo a cada `INSERT`; por eso se crean sólo los que sirven a una consulta concreta,
se usa BRIN donde alcanza y el índice parcial no se toca para envíos ya cerrados.

## Índices aplicados al sistema real

Los mismos índices sobre los fragmentos de `envios` (cada nodo indexa su fragmento).

In [ ]:
indices.eliminar_tabla_bench()
conexion.close()
for nodo, tabla in (("centro", "envios_centro"), ("norte", "envios_norte"), ("sur", "envios_sur")):
    with conectar_pg(nodo, autocommit=True) as c:
        indices.aplicar_indices(c, tabla)
        c.execute(f"ANALYZE {tabla}")
        print(nodo, [i["indice"] for i in indices.tamanios_indices(c, tabla)])

with conectar_pg("centro") as c:
    cliente = c.execute("SELECT cliente_id FROM envios_centro LIMIT 1").fetchone()["cliente_id"]
    print(
        indices.explicar(
            c,
            "SELECT * FROM envios_centro WHERE cliente_id = %s ORDER BY fecha_alta DESC LIMIT 10",
            (cliente,),
        )
    )